# 13.8 · Stable Diffusion 工作原理 / How Stable Diffusion Works

> **课程定位 / Where this fits**
> 第 8 课，**Part 13 · 生成模型**。把前面的零件拼成改变世界的**文生图**系统。
> Lesson 8, **Part 13 · Generative Models**. Assembling earlier pieces into the world-changing **text-to-image** system.
>
> **Stable Diffusion(SD)** 是开源文生图的代表作。它不是全新发明, 而是把我们学过的东西**巧妙组合**:**扩散模型(13.7)** + **自编码器/VAE(13.1-13.2)** + **CLIP(10.10/12.4)**。三大关键设计让它又快又强:①在 **VAE 的潜空间**(而非像素空间)做扩散——计算量降几十倍, 这是"Stable Diffusion"能在消费级显卡跑的根本原因;②用 **U-Net** 当去噪网络;③用 **CLIP 文本编码 + 交叉注意力**做条件, 实现"按文字生成图"。本课讲清这套架构如何协同, 并用小实验展示"潜空间扩散"为何省算力。
> **Stable Diffusion (SD)** is the flagship open-source text-to-image model. Not a brand-new invention but a clever **combination** of things we've learned: **diffusion (13.7)** + **autoencoder/VAE (13.1-13.2)** + **CLIP (10.10/12.4)**. Three key designs make it fast and powerful: ① diffuse in the **VAE's latent space** (not pixels) — tens-of-times less compute, the reason SD runs on consumer GPUs; ② a **U-Net** denoiser; ③ **CLIP text encoding + cross-attention** for conditioning, enabling "generate an image from text." We explain how this architecture cooperates and demonstrate why latent diffusion saves compute.
>
> 💼 **实战/面试视角**："潜空间扩散为什么省算力 / U-Net 的作用 / 文本条件怎么注入(交叉注意力) / classifier-free guidance / SD 三大组件" 是文生图岗高频。
> 💼 **Practical/interview angle:** "why latent diffusion saves compute / role of U-Net / how text conditions (cross-attention) / classifier-free guidance / SD's three components" — frequent for text-to-image roles.

> 📐 **符号约定 / Notation**
> - 潜空间 latent —— VAE 压缩后的低维表示(扩散在此进行) / VAE-compressed low-dim space (where diffusion happens)
> - CFG —— classifier-free guidance, 控制"多听文字" / guidance scale

> 💡 **面试相关 / Interview-relevant**
> - "Stable Diffusion 的三大组件"（出镜率 ★★★★★）
> - "潜空间扩散 vs 像素扩散(为什么用前者)"（★★★★★）
> - "文本条件怎么进入 U-Net(交叉注意力)"（★★★★）
> - "classifier-free guidance 是什么"（★★★★）
> - "U-Net 为什么适合做去噪网络"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解 SD = 扩散 + VAE + CLIP 的组合。
   Understand SD = diffusion + VAE + CLIP.
2. 理解**潜空间扩散**为何大幅省算力(动手算)。
   Understand why latent diffusion saves compute (hands-on).
3. 理解 U-Net 去噪 + CLIP 文本条件(交叉注意力)。
   Understand the U-Net denoiser + CLIP text conditioning.
4. 串起完整文生图流程, 了解 CFG。
   Put together the full text-to-image pipeline; know CFG.

## 目录 / TOC
1. [SD = 三个零件的组合 ⭐](#1)
2. [潜空间扩散：为什么省算力（动手）⭐](#2)
3. [U-Net 与文本条件(交叉注意力) ⭐](#3)
4. [完整流程 + classifier-free guidance + 小结 ⭐](#4)


<a id="1"></a>
## 1. SD = 三个零件的组合 ⭐ / SD = Three Pieces Combined

13.7 的 DDPM 直接在**像素空间**扩散——512×512×3 ≈ 78 万个数, 每步去噪都要在这么大的图上跑一遍大网络, **极其昂贵**(早期像素扩散需要大量 GPU)。Stable Diffusion 的核心贡献是**让扩散变得"亲民"**, 靠三个零件组合(面试必背):
The DDPM in 13.7 diffuses directly in **pixel space** — 512×512×3 ≈ 780k numbers, and each denoising step runs a big network over this huge image, **extremely expensive** (early pixel diffusion needed many GPUs). SD's core contribution is making diffusion **affordable**, via three pieces (must-know):
1. **VAE(自编码器)**:把图像**压缩到小得多的潜空间**(如 512×512×3 → 64×64×4), 扩散在**潜空间**里进行, 最后再解码回像素。这是"Latent Diffusion"。
   **VAE:** compress the image into a much smaller **latent space** (e.g. 512×512×3 → 64×64×4); diffusion runs **in the latent**, then decodes back to pixels. This is "Latent Diffusion."
2. **U-Net**:潜空间里的**去噪网络**(预测噪声), 带下采样-上采样+跳连(就是 10.7 的 U-Net)。
   **U-Net:** the **denoiser** in latent space (predicts noise), with down/up-sampling + skips (the U-Net from 10.7).
3. **CLIP 文本编码器**:把**文字 prompt** 编码成向量, 通过**交叉注意力**注入 U-Net, 引导生成符合文字的图。
   **CLIP text encoder:** encode the **text prompt** into vectors, injected into the U-Net via **cross-attention** to steer generation toward the text.

一句话:**用 CLIP 把文字变成条件, 用 U-Net 在 VAE 的潜空间里做条件扩散去噪, 再用 VAE 解码出图**。
In one line: **CLIP turns text into a condition; the U-Net does conditional diffusion-denoising in the VAE's latent; the VAE decodes the result into an image.**


<a id="2"></a>
## 2. 潜空间扩散：为什么省算力（动手）⭐ / Latent Diffusion: Why It Saves Compute

SD 最关键的效率创新:**不在像素空间扩散, 而在 VAE 压缩后的潜空间扩散**。直觉:图像有大量冗余(相邻像素相似), VAE 能把 512×512 的图无损感知地压成 64×64 的潜表示(空间各缩 8 倍), **元素数减少约 48 倍**。在小得多的潜空间里跑扩散, 每一步、每一次去噪都便宜几十倍。
SD's key efficiency innovation: **diffuse not in pixels but in the VAE-compressed latent space.** Intuition: images are highly redundant (neighboring pixels are similar); a VAE compresses a 512×512 image into a perceptually-near-lossless 64×64 latent (8× per spatial dim), **~48× fewer elements**. Running diffusion in this much smaller latent makes every step tens-of-times cheaper.

下面动手算一算这个节省。
Let's compute this saving hands-on.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid")

# 像素空间 vs 潜空间 的元素数对比 / element count: pixel vs latent
H = W = 512
pixel_elems = 3 * H * W                                   # RGB 像素 / RGB pixels
latent_elems = 4 * (H//8) * (W//8)                       # VAE 潜空间(下采样8倍, 4通道) / VAE latent (8x down, 4ch)
print(f"像素空间: 3×{H}×{W} = {pixel_elems:,} 个数")
print(f"潜空间:   4×{H//8}×{W//8} = {latent_elems:,} 个数")
print(f"潜空间元素数仅为像素的 1/{pixel_elems//latent_elems} → 扩散每一步便宜约 {pixel_elems//latent_elems} 倍!")

# 扩散需要跑 T 步去噪, 节省被放大 / diffusion runs T denoising steps, savings compound
T = 50
fig, ax = plt.subplots(figsize=(7,4))
res = [256, 512, 768, 1024]
pix = [3*r*r*T/1e9 for r in res]; lat = [4*(r//8)*(r//8)*T/1e9 for r in res]
ax.plot(res, pix, "s-", color="#e67", label="像素空间扩散")
ax.plot(res, lat, "o-", color="#39c", label="潜空间扩散(SD)")
ax.set_xlabel("图像分辨率"); ax.set_ylabel("总处理元素数 (×10⁹, T=50步)"); ax.legend()
ax.set_title("潜空间扩散大幅降低计算量(分辨率越高省得越多)")
plt.tight_layout(); plt.show()
print("\n这就是为什么叫'Stable Diffusion'能在单张消费级GPU上跑: 把昂贵的扩散搬到便宜的潜空间")
print("流程: 训练时VAE编码器把图→潜表示(扩散在此); 生成完用VAE解码器把潜表示→像素图")


<a id="3"></a>
## 3. U-Net 与文本条件(交叉注意力) ⭐ / U-Net & Text Conditioning

**去噪网络用 U-Net**(10.7 学过)。为什么 U-Net 适合(面试):去噪要**既看全局(整体结构、语义)又看局部(细节、纹理)**;U-Net 的编码器下采样抓全局、解码器上采样恢复细节、跳连把高分辨率细节送回去——正好同时兼顾。它还接收**时间步 $t$**(像 13.7 的时间嵌入)。
The denoiser is a **U-Net** (from 10.7). Why U-Net fits (interview): denoising needs **both global (overall structure/semantics) and local (details/texture)** views; the U-Net's encoder downsamples for global, decoder upsamples for detail, and skips pass high-res detail back — covering both. It also takes the **timestep $t$** (time embedding like 13.7).

**文字怎么进入去噪?**(SD 的灵魂)用**交叉注意力(cross-attention, 12.6 学过)**:
**How does text enter denoising?** (SD's soul) via **cross-attention (from 12.6):**
- CLIP 文本编码器把 prompt(如 "a cat in space")编码成一串文本向量。
  The CLIP text encoder encodes the prompt (e.g. "a cat in space") into text vectors.
- 在 U-Net 的多个层里插入**交叉注意力**:**Query 来自图像特征, Key/Value 来自文本向量**。于是去噪网络在每一步都"**回看文字**", 让生成的内容**对齐 prompt**——要画猫的地方关注"cat"那个词, 要画太空背景的地方关注"space"。
  Cross-attention is inserted in several U-Net layers: **Query from image features, Key/Value from text vectors.** So the denoiser "**looks at the text**" at each step, aligning content with the prompt — attending to "cat" where a cat should go, "space" for the background.

这正是 12.2/12.6 注意力机制的直接应用:**用文本作为条件, 通过注意力控制生成内容**。
This is a direct application of the attention from 12.2/12.6: **use text as a condition and control generation via attention.**


<a id="4"></a>
## 4. 完整流程 + classifier-free guidance + 小结 ⭐ / Full Pipeline & CFG

**完整文生图流程**(把所有零件串起来):
The **full text-to-image pipeline** (all pieces together):
```
文字 prompt ──[CLIP 文本编码器]──> 文本向量 ┐
                                            │ (交叉注意力注入)
纯噪声(潜空间) ──[U-Net 去噪 ×N步, 看t和文本]──> 干净的潜表示 ──[VAE 解码器]──> 最终图像
pure noise (latent) → U-Net denoise ×N steps (conditioned on t & text) → clean latent → VAE decoder → image
```
从潜空间的纯噪声开始, U-Net 在文本条件下反复去噪(如 20-50 步, 用 DDIM 加速), 得到干净的潜表示, 最后 VAE 解码成像素图。
Start from pure noise in latent space; the U-Net denoises repeatedly under text conditioning (e.g. 20–50 steps with DDIM), yielding a clean latent, finally decoded to pixels by the VAE.

**Classifier-Free Guidance(CFG)**(面试高频):一个控制"**多听文字**"的关键技巧。每步同时算**有文本条件**和**无条件**两个噪声预测, 然后**朝"文本条件比无条件更强调的方向"放大**:
**Classifier-Free Guidance (CFG)** (high-frequency): a key trick controlling "**how much to obey the text**." Each step computes both a **text-conditioned** and an **unconditional** noise prediction, then **amplifies the direction the conditioned one favors**:

$$\epsilon = \epsilon_{\text{uncond}} + w \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$$

引导强度 $w$ 越大→越严格贴合 prompt(但太大会失真/不自然);$w$ 小→更自由多样但可能跑题。这就是 SD 里那个 "guidance scale" 参数。
A larger guidance scale $w$ → stricter prompt adherence (too large distorts); smaller $w$ → freer/diverse but may drift. This is SD's "guidance scale" parameter.

```
Stable Diffusion = 扩散(13.7) + VAE(13.1) + CLIP(12.4) 的组合
①潜空间扩散: VAE把图压到小潜空间(512²→64², 元素少~48倍), 扩散在此进行→省几十倍算力(SD能上消费级GPU的关键)
②U-Net去噪: 编码器抓全局+解码器恢复细节+跳连; 接收时间步t
③文本条件: CLIP编码prompt → 交叉注意力(Q=图像,K/V=文本)注入U-Net → 生成对齐文字
流程: 文字→CLIP→ [潜空间纯噪声→U-Net条件去噪N步→干净潜表示] →VAE解码→图
CFG(classifier-free guidance): ε=ε_uncond+w(ε_cond-ε_uncond); w大=更贴prompt(太大失真)
```

### 💡 面试速查 / Interview cheat-sheet
1. **SD三组件**: VAE(潜空间压缩) + U-Net(去噪) + CLIP(文本条件)。
   SD's three: VAE (latent compression) + U-Net (denoiser) + CLIP (text conditioning).
2. **潜空间扩散**: 在VAE小潜空间扩散而非像素, 省几十倍算力(上消费级GPU)。
   Latent diffusion: diffuse in the small VAE latent, not pixels; tens-× cheaper.
3. **文本注入**: CLIP编码prompt, 交叉注意力(Q图像/KV文本)进U-Net。
   Text injection: CLIP encodes prompt, cross-attention (Q image / KV text) into U-Net.
4. **U-Net**: 全局+局部兼顾, 适合去噪; 带时间步t。
   U-Net: global+local, suits denoising; takes timestep t.
5. **CFG**: ε_uncond+w(ε_cond-ε_uncond); w控制贴合prompt程度。
   CFG: ε_uncond+w(ε_cond-ε_uncond); w controls prompt adherence.

### 下一节 / Next
**13.9 评估生成模型**——生成的图好不好、多样不多样, 怎么客观衡量? 我们会**从零实现 FID(Fréchet Inception 距离)** 和 **Inception Score(IS)**, 理解它们如何同时衡量"保真度"和"多样性", 以及各自的坑。
**13.9 Evaluating Generative Models** — how to objectively measure if generated images are good and diverse? We'll **implement FID (Fréchet Inception Distance)** and **Inception Score (IS)** from scratch, understand how they capture "fidelity" and "diversity," and their pitfalls.
